# Customer Churn Prediction — Step-by-Step
This notebook predicts whether a telecom customer will **churn** (leave the company) using their account data.

**How to use this notebook:**
1. Open [Google Colab](https://colab.research.google.com) → File → Upload Notebook → upload this `.ipynb` file.
2. Follow Step 1 below to get the dataset, then run each cell top-to-bottom (Shift+Enter).

No prior ML experience needed — every cell has comments explaining what it does and why.

## Step 1: Get the dataset

1. Go to [Kaggle: Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) (you may need a free Kaggle account).
2. Download `WA_Fn-UseC_-Telco-Customer-Churn.csv`.
3. In Colab, click the **folder icon** on the left sidebar → click the **upload icon** → select the CSV.
4. Run the cell below to confirm it loaded correctly.

In [ ]:
# Step 1: Load the dataset
import pandas as pd

# Change this filename if yours is different
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape:", df.shape)   # (rows, columns)
df.head()   # preview the first 5 rows

## Step 2: Explore the data (EDA)

Before building any model, we look at the data to understand it: what columns exist, how many customers churned, and which factors seem related to churn.

In [ ]:
# Step 2a: Basic info — column names, data types, missing values
df.info()

In [ ]:
# Step 2b: How many customers churned vs stayed?
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True) * 100)  # as percentages

In [ ]:
# Step 2c: Visualize churn by contract type
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title("Churn by Contract Type")
plt.xticks(rotation=15)
plt.show()

# Insight to look for: month-to-month customers usually churn far more
# than customers on 1-year or 2-year contracts.

In [ ]:
# Step 2d: Visualize churn by tenure (how long they've been a customer)
plt.figure(figsize=(6,4))
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', bins=30)
plt.title("Churn by Tenure (months)")
plt.show()

# Insight to look for: newer customers (low tenure) usually churn more —
# this is common in subscription businesses.

## Step 3: Clean the data

Real datasets are messy. Here we:
- Fix a column that's stored as text but should be numeric (`TotalCharges`)
- Remove an ID column that has no predictive value
- Convert categorical (text) columns into numbers, since ML models only understand numbers

In [ ]:
# Step 3a: Fix TotalCharges — it's stored as text and has some blank values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Missing values after conversion:", df['TotalCharges'].isna().sum())

# Fill missing values with the median (a safe, simple default)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

In [ ]:
# Step 3b: Drop customerID — it's a unique identifier, not a useful predictor
df = df.drop(columns=['customerID'])

# Step 3c: Convert the target column Churn (Yes/No) into 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
# Step 3d: Convert all other text (categorical) columns into numeric columns
# get_dummies creates a separate 0/1 column for each category (called "one-hot encoding")
df_encoded = pd.get_dummies(df, drop_first=True)

print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

## Step 4: Split the data and train two models

We split the data into a **training set** (model learns from this) and a **test set**
(model has never seen this — used to check if it actually learned something useful).

We'll train two models so you can compare them:
- **Logistic Regression** — simple, fast, easy to explain
- **Random Forest** — usually more accurate, and tells us which features mattered most

In [ ]:
# Step 4a: Split into features (X) and target (y), then train/test
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

In [ ]:
# Step 4b: Train Logistic Regression
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

log_preds = log_model.predict(X_test)
print("Logistic Regression trained.")

In [ ]:
# Step 4c: Train Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
print("Random Forest trained.")

## Step 5: Evaluate both models

We check:
- **Accuracy** — % of predictions that were correct overall
- **Precision** — of customers we predicted would churn, how many actually did
- **Recall** — of customers who actually churned, how many did we catch
- **F1-score** — a balance between precision and recall

For churn prediction, **recall** often matters most — missing a customer who's about to leave is usually costlier than a false alarm.

In [ ]:
# Step 5a: Compare both models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def report(name, y_true, y_pred):
    print(f"--- {name} ---")
    print("Accuracy: ", round(accuracy_score(y_true, y_pred), 3))
    print("Precision:", round(precision_score(y_true, y_pred), 3))
    print("Recall:   ", round(recall_score(y_true, y_pred), 3))
    print("F1-score: ", round(f1_score(y_true, y_pred), 3))
    print()

report("Logistic Regression", y_test, log_preds)
report("Random Forest", y_test, rf_preds)

In [ ]:
# Step 5b: Confusion matrix — visualize correct vs incorrect predictions
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(10,4))
ConfusionMatrixDisplay.from_predictions(y_test, log_preds, ax=axes[0])
axes[0].set_title("Logistic Regression")
ConfusionMatrixDisplay.from_predictions(y_test, rf_preds, ax=axes[1])
axes[1].set_title("Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
# Step 5c: Which features matter most? (Random Forest only)
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(7,5))
top_features.plot(kind='barh')
plt.title("Top 10 Features Driving Churn")
plt.gca().invert_yaxis()
plt.show()

print(top_features)

## Step 6: Write up your findings (for your resume/GitHub README)

Fill in the blanks once you've run everything above:

- **Dataset:** Telco Customer Churn (~7,000 customers, Kaggle)
- **Models compared:** Logistic Regression vs Random Forest
- **Best model:** _______ with accuracy of ___%, recall of ___%
- **Top churn drivers found:** _______, _______, _______ (from Step 5c)
- **Business takeaway:** e.g. "Month-to-month contract customers with low tenure are the highest churn risk — targeted retention offers for this segment could reduce churn."

## Step 7: Push to GitHub

```bash
git init
git add .
git commit -m "Customer churn prediction using Logistic Regression and Random Forest"
git remote add origin https://github.com/<your-username>/customer-churn-prediction.git
git push -u origin main
```

Add a short `README.md` with the findings from Step 6 — that's what recruiters actually read first.